# 固定模型的推理成本对照

比较 B06、DistilBERT、TextCNN + DistilBERT 和完整加权融合。使用同一批从验证集固定抽取的评论，分别以批大小 1 和 16 运行。双模型沿用 Notebook 024 的权重 (0.375, 0.625)，完整融合沿用 Notebook 023 的权重 (0.2, 0.3, 0.5)。

每次从评论字符串开始，计入分词、特征或张量构造、设备传输、模型前向、投票和返回 CPU 标签。B06 使用 CPU，TextCNN 与 DistilBERT 使用同一块 GPU，融合顺序执行各模型。GPU 结果取回 CPU 后才停止计时。模型加载、数据读取与预热单独进行。每轮更换方案的测量顺序以减小顺序影响。

In [ ]:
# 核对输入并加载固定模型
import hashlib
import json
import os
import platform
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from sentiment_analysis.paths import ROOT, CONFIGS_DIR, DATA_DIR, TABLES_DIR
from sentiment_analysis.sequence_data import TOKEN_PATTERN, prepare_sequence
from sentiment_analysis.textcnn import TextCNN


def fixed_path(record):
    path = ROOT / record["path"]
    with path.open("rb") as file:
        digest = hashlib.file_digest(file, "sha256").hexdigest()
    if digest != record["sha256"]:
        raise ValueError(f"Fixed file changed: {path.name}")
    return path


protocol = json.loads((CONFIGS_DIR / "final_test_protocol.json").read_text(encoding="utf-8"))
voting = json.loads((CONFIGS_DIR / "three_model_voting_test_protocol.json").read_text(encoding="utf-8"))
distilbert = json.loads((CONFIGS_DIR / "distilbert_test_protocol.json").read_text(encoding="utf-8"))
validation_data = pd.read_csv(fixed_path(protocol["files"]["validation"]))

b06 = joblib.load(fixed_path(voting["files"]["calibrated_b06"]))
svm_model = b06.calibrated_classifiers_[0].estimator
positive_label = voting["positive_label"]
positive_index = list(b06.classes_).index(positive_label)

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("This benchmark uses CUDA with BF16 support.")
device = torch.device("cuda:0")
cnn = torch.load(fixed_path(protocol["files"]["textcnn"]), map_location="cpu", weights_only=True)
if cnn["token_pattern"] != TOKEN_PATTERN.pattern or not cnn["lowercase"]:
    raise ValueError("TextCNN preprocessing does not match the checkpoint.")
cnn_model = TextCNN(**cnn["model_config"]).to(device)
cnn_model.load_state_dict(cnn["model_state_dict"])
cnn_model.eval()

model_dir = DATA_DIR / "models" / "distilbert_seed42"
for name, digest in distilbert["checkpoint_sha256"].items():
    fixed_path({"path": (model_dir / name).relative_to(ROOT), "sha256": digest})
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, trust_remote_code=False)
model = AutoModelForSequenceClassification.from_pretrained(
    model_dir, local_files_only=True, use_safetensors=True, trust_remote_code=False,
    attn_implementation=distilbert["attention_implementation"],
).to(device)
model.eval()
if model.config.id2label != {0: "NEGATIVE", 1: "POSITIVE"}:
    raise ValueError("DistilBERT label mapping does not match the checkpoint.")

full_weights = np.array(voting["soft_weights"]["Weighted soft vote"])
pair_weights = full_weights[1:] / full_weights[1:].sum()
threshold = voting["soft_positive_threshold"]
print(f"Models loaded on CPU and {torch.cuda.get_device_name(device)}.", flush=True)

In [ ]:
# 从评论文本生成各方案的类别预测
def predict_b06(texts):
    return svm_model.predict(texts)


def textcnn_probability(texts):
    sequences = [prepare_sequence(text, cnn["vocabulary"], cnn["max_length"]) for text in texts]
    token_ids, lengths = zip(*sequences)
    token_ids = torch.tensor(token_ids, dtype=torch.long).to(device)
    lengths = torch.tensor(lengths, dtype=torch.long).to(device)
    logits = cnn_model(token_ids, lengths)
    return logits.softmax(dim=1)[:, positive_label]


def distilbert_logits(texts):
    batch = tokenizer(
        texts, truncation=True, max_length=distilbert["max_length"], padding=True,
        pad_to_multiple_of=distilbert["pad_to_multiple_of"], return_tensors="pt",
        return_token_type_ids=False,
    ).to(device)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        return model(**batch).logits


@torch.inference_mode()
def predict_distilbert(texts):
    return distilbert_logits(texts).argmax(dim=1).cpu().numpy()


@torch.inference_mode()
def neural_probabilities(texts):
    cnn_probability = textcnn_probability(texts).cpu().numpy()
    bert_probability = distilbert_logits(texts).float().softmax(dim=1)[:, positive_label].cpu().numpy()
    return cnn_probability, bert_probability


def predict_pair(texts):
    probabilities = np.column_stack(neural_probabilities(texts))
    return (probabilities @ pair_weights > threshold).astype(int)


def predict_full(texts):
    b06_probability = b06.predict_proba(texts)[:, positive_index]
    probabilities = np.column_stack([b06_probability, *neural_probabilities(texts)])
    return (probabilities @ full_weights > threshold).astype(int)


predictors = {
    "B06": ("CPU", predict_b06),
    "DistilBERT": ("GPU", predict_distilbert),
    "TextCNN + DistilBERT": ("GPU", predict_pair),
    "Full weighted vote": ("CPU + GPU", predict_full),
}

In [ ]:
# 用原验证预测核对同一批文本上的推理输出
validation_predictions = pd.read_csv(
    TABLES_DIR / "three_model_soft_voting_validation_predictions.csv", float_precision="round_trip",
)
np.testing.assert_array_equal(validation_predictions["VALIDATION ROW"], np.arange(len(validation_data)))
np.testing.assert_array_equal(validation_predictions["LABEL"], validation_data["label"])
check_size = distilbert["batch_size"]
check_texts = validation_data["text"].iloc[:check_size].tolist()
expected = validation_predictions.iloc[:check_size][[
    "B06 POSITIVE PROBABILITY", "TEXTCNN POSITIVE PROBABILITY", "DISTILBERT POSITIVE PROBABILITY",
]].to_numpy()
actual = np.column_stack([b06.predict_proba(check_texts)[:, positive_index], *neural_probabilities(check_texts)])
np.testing.assert_allclose(actual, expected, rtol=0, atol=1e-5)
np.testing.assert_array_equal(predict_distilbert(check_texts), expected[:, 2] > threshold)
np.testing.assert_array_equal(predict_pair(check_texts), expected[:, 1:] @ pair_weights > threshold)
np.testing.assert_array_equal(predict_full(check_texts), expected @ full_weights > threshold)
print("Validation inference verified against Notebook 022.", flush=True)

In [ ]:
# 每个方案在相同样本上重复计时
seed = 42
sample_rows = np.random.default_rng(seed).choice(len(validation_data), size=256, replace=False)
texts = validation_data["text"].iloc[sample_rows].tolist()
repeats = 4
warmup_batches = 4
batch_sizes = [1, distilbert["batch_size"]]
measurements = []
for batch_size in batch_sizes:
    batches = [texts[start:start + batch_size] for start in range(0, len(texts), batch_size)]
    names = list(predictors)
    durations = {name: [] for name in names}
    for repeat in range(repeats):
        offset = repeat % len(names)
        for name in names[offset:] + names[:offset]:
            _, predict = predictors[name]
            for batch in batches[:warmup_batches]:
                predict(batch)
            for batch in batches:
                start = perf_counter()
                prediction = predict(batch)
                elapsed = perf_counter() - start
                if prediction.shape != (len(batch),) or not np.isin(prediction, [0, 1]).all():
                    raise ValueError(f"Invalid predictions: {name}")
                durations[name].append(elapsed)
        print(f"Batch size {batch_size} | repeat {repeat + 1}/{repeats}", flush=True)

    for name, elapsed in durations.items():
        device_name, _ = predictors[name]
        measurements.append({
            "MODEL": name,
            "DEVICE": device_name,
            "BATCH SIZE": batch_size,
            "TIMED REQUESTS": len(elapsed),
            "MEDIAN REQUEST LATENCY (ms)": 1000 * np.median(elapsed),
            "P95 REQUEST LATENCY (ms)": 1000 * np.percentile(elapsed, 95),
            "THROUGHPUT (reviews/s)": repeats * len(texts) / sum(elapsed),
        })
benchmark = pd.DataFrame(measurements)

In [ ]:
# 附上已有评估分数，并保存耗时及运行环境
single_scores = pd.read_csv(TABLES_DIR / "distilbert_test_comparison.csv").set_index("MODEL")
full_test = pd.read_csv(TABLES_DIR / "three_model_voting_test_comparison.csv").set_index("MODEL")
ablation = pd.read_csv(TABLES_DIR / "three_model_voting_ablation_validation.csv").set_index("MODEL")
scores = single_scores.loc[["B06", "DistilBERT"], ["VALIDATION F1", "TEST F1"]].copy()
scores.loc["TextCNN + DistilBERT"] = [ablation.loc["Without B06", "F1"], np.nan]
scores.loc["Full weighted vote"] = [ablation.loc["Full weighted vote", "F1"], full_test.loc["Weighted soft vote", "F1"]]
comparison = benchmark.join(scores, on="MODEL")

cpu = platform.processor()
if platform.system() == "Windows":
    import winreg

    with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0") as key:
        cpu = winreg.QueryValueEx(key, "ProcessorNameString")[0].strip()

metadata = {
    "cpu": cpu,
    "logical_cpus": os.cpu_count(),
    "gpu": torch.cuda.get_device_name(device),
    "platform": platform.platform(),
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "torch_cpu_threads": torch.get_num_threads(),
    "transformers": transformers.__version__,
    "scikit_learn": sklearn.__version__,
    "validation_sha256": protocol["files"]["validation"]["sha256"],
    "sample_seed": seed,
    "validation_rows": sample_rows.tolist(),
    "warmup_requests_per_repeat": warmup_batches,
    "repeats": repeats,
}
comparison.to_csv(TABLES_DIR / "inference_benchmark_comparison.csv", index=False)
(TABLES_DIR / "inference_benchmark_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n", encoding="utf-8",
)
display(comparison.round(4))

## 实测结果

在 13th Gen Intel(R) Core(TM) i9-13900HX 和 NVIDIA GeForce RTX 4060 Laptop GPU 上，每个方案使用相同的 256 条验证评论，各测四轮。单条延迟统计每条请求的耗时，批量吞吐量按连续处理的评论数除以对应请求总耗时计算。以下 F1 来自此前对完整划分的评估。

| 方案 | 单条中位延迟 | 单条 P95 延迟 | 批量吞吐量（16 条/批） | 验证 F1 | 测试 F1 |
| --- | ---: | ---: | ---: | ---: | ---: |
| B06 | 0.59 ms | 1.22 ms | 3,165 条/s | 88.89% | 89.01% |
| DistilBERT | 7.32 ms | 10.93 ms | 582 条/s | 90.88% | 90.90% |
| TextCNN + DistilBERT | 8.36 ms | 12.49 ms | 539 条/s | 92.17% | — |
| 完整加权融合 | 9.79 ms | 14.75 ms | 432 条/s | 92.87% | 92.25% |

相对 DistilBERT 单模型，完整融合的测试 F1 高 1.34 个百分点，单条中位延迟增加约 34%，批量吞吐量下降约 26%。TextCNN + DistilBERT 的开销介于两者之间，其 F1 来自 Notebook 024 的验证集。耗时对应这台机器上的 CPU/GPU 分工和顺序执行方式。
